# Task: Breast Cancer Classification

Goal: predict if a tumor is **malignant** (cancer) or **benign** (not cancer) from measurements like radius, texture and area.

Models:
- Logistic Regression, Decision Tree, Random Forest, AdaBoost
- KNN and SVM without grid search
- KNN and SVM with grid search

## Load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

print("Shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Classes:", dict(enumerate(data.target_names)))

df.head()

In [ ]:
# class balance
df["target"].value_counts(normalize=True)

- There are 569 rows and 30 numeric features, with no missing values, so no cleaning is needed.
- Class `0` = malignant (about 37%), class `1` = benign (about 63%).
- In cancer detection, the important thing is to catch the **malignant** cases, so we will look at F1 for class `0`, not only accuracy.

## Split and scale

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("target", axis=1)
y = df["target"]

# stratify keeps the same class ratio in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# scaled version (for Logistic Regression, KNN and SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Tree models (Decision Tree, Random Forest, AdaBoost) don't care about scale, so they use the original data. Logistic Regression, KNN and SVM use the scaled data.

## Logistic Regression, Decision Tree, Random Forest, AdaBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

results = {}   # to compare all models at the end

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

for name, model in models.items():
    print(f"--- Model: {name} ---")

    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, pos_label=0)   # F1 for malignant
    results[name] = {"Accuracy": acc, "F1 (malignant)": f1}

    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=data.target_names))
    print("="*50)

- **Logistic Regression**: simple linear model.
- **Decision Tree**: one tree, fast but can overfit.
- **Random Forest**: many trees voting together (bagging).
- **AdaBoost**: small trees trained one after another, each one focuses on the mistakes of the previous one (boosting).

## KNN & SVM without grid search

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

models_default = {
    "SVM (Default)": SVC(random_state=42),
    "KNN (Default)": KNeighborsClassifier()
}

for name, model in models_default.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, pos_label=0)
    results[name] = {"Accuracy": acc, "F1 (malignant)": f1}

    print(f"Model: {name}")
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=data.target_names))
    print("-" * 50)

Defaults: KNN uses `n_neighbors=5`, SVM uses `kernel='rbf'` and `C=1`.

## Elbow method for KNN

We try different `k` values and plot the error rate. (This is only to see the shape of the curve, the real choice of `k` is made by grid search below using cross validation.)

In [ ]:
error_rate = []
k_values = range(1, 21)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    error_rate.append(1 - accuracy_score(y_test, y_pred))

plt.figure(figsize=(10, 5))
plt.plot(k_values, error_rate, marker="o")
plt.xlabel("K Value")
plt.ylabel("Error Rate")
plt.title("Elbow Method for KNN")
plt.xticks(k_values)
plt.show()

## Effect of C in SVM (soft vs hard margin)

- Small `C` = soft margin (allows mistakes, simpler model).
- Big `C` = hard margin (tries to classify every training point correctly, can overfit).

If train accuracy keeps going up but test accuracy doesn't, that means overfitting.

In [ ]:
c_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
c_results = []

for c in c_values:
    svm_model = SVC(C=c, random_state=42)
    svm_model.fit(X_train_scaled, y_train)

    c_results.append({
        "C": c,
        "train_accuracy": accuracy_score(y_train, svm_model.predict(X_train_scaled)),
        "test_accuracy": accuracy_score(y_test, svm_model.predict(X_test_scaled))
    })

pd.DataFrame(c_results)

## KNN & SVM with grid search

Grid search tries every combination of the parameters and picks the best one using cross validation (`cv=5`).

In [ ]:
svm_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 0.001, 0.01, 0.1]
}

knn_param_grid = {
    'n_neighbors': list(range(3, 16, 2)),
    'weights': ['uniform', 'distance']   # distance: closer neighbors have more vote
}

svm_grid = GridSearchCV(SVC(random_state=42), svm_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train_scaled, y_train)

knn_grid = GridSearchCV(KNeighborsClassifier(), knn_param_grid, cv=5, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train_scaled, y_train)

print(f"SVM: {svm_grid.best_params_}")
svm_best_pred = svm_grid.predict(X_test_scaled)
print(f"Accuracy after Grid Search: {accuracy_score(y_test, svm_best_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, svm_best_pred, target_names=data.target_names))

print("-" * 50)

print(f"KNN: {knn_grid.best_params_}")
knn_best_pred = knn_grid.predict(X_test_scaled)
print(f"Accuracy after Grid Search: {accuracy_score(y_test, knn_best_pred):.4f}")
print("Classification Report:")
print(classification_report(y_test, knn_best_pred, target_names=data.target_names))
print("=" * 50)

results["SVM (Grid)"] = {"Accuracy": accuracy_score(y_test, svm_best_pred), "F1 (malignant)": f1_score(y_test, svm_best_pred, pos_label=0)}
results["KNN (Grid)"] = {"Accuracy": accuracy_score(y_test, knn_best_pred), "F1 (malignant)": f1_score(y_test, knn_best_pred, pos_label=0)}

- `gamma` (for rbf / poly): small = smooth boundary, big = complex boundary (can overfit).
- `n_neighbors`: small k = complex boundary (can overfit), big k = smoother boundary.

## Compare all models

In [ ]:
results_df = pd.DataFrame(results).T.sort_values(by="F1 (malignant)", ascending=False)
results_df.round(4)

In [ ]:
results_df.sort_values(by="F1 (malignant)").plot(kind="barh", figsize=(10, 5))
plt.xlim(0.8, 1.0)
plt.title("Model Comparison")
plt.show()

## Confusion matrix for the best model

Rows = real class, columns = predicted class. The most dangerous mistake here is predicting **benign** when the tumor is really **malignant** (false negative).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

best_name = results_df.index[0]
print("Best model (by F1 malignant):", best_name)

# get the predictions of the best model
all_models = {
    **models,
    **models_default,
    "SVM (Grid)": svm_grid.best_estimator_,
    "KNN (Grid)": knn_grid.best_estimator_
}
best_model = all_models[best_name]

if best_name in ["Decision Tree", "Random Forest", "AdaBoost"]:
    y_best = best_model.predict(X_test)
else:
    y_best = best_model.predict(X_test_scaled)

ConfusionMatrixDisplay.from_predictions(y_test, y_best, display_labels=data.target_names)
plt.show()